# Simple RAG System - Tips Hindawi University

This notebook builds a full **Retrieval-Augmented Generation (RAG)** pipeline using real models from Hugging Face:

- **Embedding model**: `sentence-transformers/all-MiniLM-L6-v2`
- **Generation model**: `google/flan-t5-large`


## 1) Install dependencies

In [ ]:
!pip install -q pypdf sentence-transformers transformers torch


## 2) Imports

In [ ]:
import re
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


## 3) Load the PDF (Loader)
Extract the raw text from the PDF that will serve as the knowledge base.

In [ ]:
def load_pdf(path: str) -> str:
    """Extract all text from a PDF file."""
    reader = PdfReader(path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text

pdf_path = "Tips Hindawi University Info.pdf"  # adjust the path if needed
raw_text = load_pdf(pdf_path)
print(raw_text[:500])


1. General Overview
Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle
East. Founded in 1963, the university has grown into a globally recognized center for academic excellence
and innovation. With over six decades of educational leadership, THU has produced more than 150,000
graduates who serve in diverse industries and academic circles worldwide.
The university is accredited by the International Commission for Academic Standards and th


## 4) Split the text into chunks
The text is split into overlapping word-based chunks, so each chunk is short enough to embed well,
and overlap prevents information from being cut off right at a chunk boundary.

In [ ]:
def chunk_text(text: str, chunk_size: int = 60, overlap: int = 15):
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print(f"Number of chunks: {len(chunks)}")
print(chunks[0])


Number of chunks: 14
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic


## 5) Embedder + Retriever
Load an embedding model from Hugging Face (`all-MiniLM-L6-v2`), embed every chunk once,
then build a retriever that compares the question against all chunks using **cosine similarity**.


In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Compute embeddings for all chunks once
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

def retrieve(query: str, top_k: int = 3, min_score: float = 0.40):
    query_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = chunk_embeddings @ query_emb  # cosine similarity (vectors are normalized)
    ranked_idx = np.argsort(scores)[::-1][:top_k]
    results = [(chunks[i], float(scores[i])) for i in ranked_idx if scores[i] >= min_score]
    return results

# quick sanity check
for chunk, score in retrieve("Where is the university located?"):
    print(f"[{score:.3f}] {chunk[:100]}...")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[0.516] an environment of research, creativity, and public service. 2. Campus and Facilities 2.1 Main Campus...
[0.500] leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academi...
[0.488] 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education locat...


## 6) Generator
Load a generative model and build a prompt containing the retrieved context plus the question,
so the model produces a natural answer instead of returning a raw extracted sentence.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def build_prompt(query: str, context: str) -> str:
    return (
        "You are a careful assistant that answers questions using ONLY the context provided.\n"
        "Rules:\n"
        "- Base your answer strictly on the context. Do not use outside knowledge.\n"
        "- Do not guess or infer anything the context does not clearly state.\n"
        "- Answer in one short sentence, in your own words. Do NOT copy the context verbatim.\n"
        "- If the context does not clearly contain the answer, respond with exactly: "
        "'Not mentioned in the document.'\n\n"
        "Example 1:\n"
        "Context: The library has over 1 million volumes and four residence halls.\n"
        "Question: Does the university have a swimming pool?\n"
        "Answer: Not mentioned in the document.\n\n"
        "Example 2:\n"
        "Context: Merit Scholarships cover 25% to 100% of tuition. Need-Based Grants are also "
        "available, along with Research Fellowships for graduate students.\n"
        "Question: What kind of financial aid does the university offer?\n"
        "Answer: The university offers merit-based scholarships, need-based grants, and research "
        "fellowships for graduate students.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

def generate_answer(query: str, retrieved: list):
    if not retrieved:
        return "Not mentioned in the document."

    context = "\n".join(chunk for chunk, _ in retrieved)
    prompt = build_prompt(query, context)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = gen_model.generate(**inputs, max_new_tokens=50)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 7) Combine into a full pipeline (Retriever + Generator)

In [ ]:
class SimpleRAG:
    def __init__(self, top_k: int = 3):
        self.top_k = top_k

    def ask(self, query: str):
        retrieved = retrieve(query, top_k=self.top_k)
        answer = generate_answer(query, retrieved)
        return answer, retrieved

rag = SimpleRAG()


## 8) Ask your own questions
Instead of a fixed list, this cell lets you type a question yourself. Type `exit` (or `quit`) to stop.

In [ ]:
while True:
    query = input("Ask a question about Tips Hindawi University (or type 'exit' to quit): ").strip()

    if query.lower() in ("exit", "quit"):
        print("Goodbye!")
        break

    if not query:
        continue

    answer, retrieved = rag.ask(query)
    print("=" * 70)
    print(f"Q: {query}")
    print(f"A: {answer}")
    print("-- retrieved chunks --")
    for chunk, score in retrieved:
        print(f"  [{score:.3f}] {chunk[:100]}...")
    print()
